# Day 050 — Exercise 1: load_and_clean

**What you'll build:** `load_and_clean(source) -> pd.DataFrame` — accept a CSV string, file path, or DataFrame; parse date columns; fill numeric NaN with the column median; drop exact duplicate rows.

**Why it matters:** Every real dataset is dirty. `load_and_clean` is the first stage of the Insight Engine. It standardises the input so the rest of the pipeline can assume no nulls in numeric columns and no duplicate rows.

## Provided: Setup + Data Generator

In [ ]:
import io
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import ollama
warnings.filterwarnings('ignore')


def make_sample_data(n: int = 300, seed: int = 42) -> pd.DataFrame:
    """
    Retail sales dataset.
    Columns: date (str), region, category, units_sold, price, discount, revenue.
    Revenue = units_sold*4 + price*1.5 - discount*150 + noise (5% nulls injected).
    """
    rng      = np.random.default_rng(seed)
    dates    = pd.date_range('2023-01-01', periods=n, freq='D').strftime('%Y-%m-%d')
    region   = rng.choice(['North', 'South', 'East', 'West'], n)
    category = rng.choice(['Electronics', 'Clothing', 'Food', 'Books'], n)
    units    = rng.integers(1, 50, n)
    price    = rng.uniform(5.0, 200.0, n).round(2)
    discount = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], n)
    revenue  = (units * 4.0 + price * 1.5 - discount * 150
                + rng.standard_normal(n) * 20).round(2)
    null_idx = rng.choice(n, size=max(1, int(n * 0.05)), replace=False)
    revenue  = revenue.astype(float)
    revenue[null_idx] = np.nan
    return pd.DataFrame({
        'date':       pd.Series(dates),
        'region':     region,
        'category':   category,
        'units_sold': units,
        'price':      price,
        'discount':   discount,
        'revenue':    revenue,
    })

## Your Implementation

In [ ]:
def load_and_clean(source) -> pd.DataFrame:
    """
    Load from CSV string / file path / DataFrame, then:
      1. Parse columns containing 'date' or 'time' in their name → datetime64
      2. Fill numeric NaN with the column median
      3. Drop exact duplicate rows
    """
    # Step 1 — Load
    if isinstance(source, pd.DataFrame):
        df = source.copy()
    elif isinstance(source, str) and ('\n' in source or ',' in source[:200]):
        df = pd.read_csv(io.StringIO(source))
    else:
        df = pd.read_csv(source)

    # Step 2 — Parse date columns (look for 'date' or 'time' in column name)
    # TODO: for col in df.columns:
    #     if any(kw in col.lower() for kw in ('date', 'time')):
    #         try:
    #             df[col] = pd.to_datetime(df[col], errors='coerce')
    #         except Exception:
    #             pass

    # Step 3 — Fill numeric NaN with median
    # TODO: for col in df.select_dtypes(include='number').columns:
    #     df[col] = df[col].fillna(df[col].median())

    # Step 4 — Drop duplicates
    # TODO: df = df.drop_duplicates().reset_index(drop=True)

    return df

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    raw = make_sample_data(200)

    # Check 1: returns DataFrame
    try:
        df = load_and_clean(raw)
        assert isinstance(df, pd.DataFrame), \
            f'expected DataFrame, got {type(df).__name__}'
        passed += 1; print(f'\u2705 Check 1: returns DataFrame with shape {df.shape}')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: no NaN in numeric columns after cleaning
    try:
        num_nulls = df.select_dtypes(include='number').isnull().sum().sum()
        assert num_nulls == 0, \
            f'expected 0 numeric nulls after cleaning, got {num_nulls}'
        passed += 1; print(f'\u2705 Check 2: 0 numeric nulls after cleaning')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: no duplicate rows
    try:
        dupes = int(df.duplicated().sum())
        assert dupes == 0, f'expected 0 duplicates, got {dupes}'
        passed += 1; print(f'\u2705 Check 3: 0 duplicate rows')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: date column is datetime64
    try:
        assert 'date' in df.columns, 'expected a date column'
        assert pd.api.types.is_datetime64_any_dtype(df['date']), \
            f"date column dtype is {df['date'].dtype}, expected datetime64"
        passed += 1; print(f'\u2705 Check 4: date column is datetime64')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: works with CSV string input too
    try:
        csv_text = raw.to_csv(index=False)
        df2 = load_and_clean(csv_text)
        assert isinstance(df2, pd.DataFrame)
        assert df2.select_dtypes(include='number').isnull().sum().sum() == 0
        passed += 1; print(f'\u2705 Check 5: CSV string input works')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def load_and_clean(source) -> pd.DataFrame:
    """
    Load from CSV string / file path / DataFrame and clean.

    Steps applied in order:
      1. Parse source into a DataFrame
      2. Detect and parse date/time columns to datetime64
      3. Fill numeric NaN with column median
      4. Drop exact duplicate rows
    """
    if isinstance(source, pd.DataFrame):
        df = source.copy()
    elif isinstance(source, str) and ('\n' in source or ',' in source[:200]):
        df = pd.read_csv(io.StringIO(source))
    else:
        df = pd.read_csv(source)

    # Detect date columns by name
    for col in df.columns:
        if any(kw in col.lower() for kw in ('date', 'time', 'created', 'updated')):
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
            except Exception:
                pass

    # Fill numeric NaN with column median
    for col in df.select_dtypes(include='number').columns:
        median = df[col].median()
        df[col] = df[col].fillna(median)

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)
    return df
```

</details>